In [0]:
# Check available silver tables
display(spark.sql("""SHOW TABLES IN project.silver"""))

## Helper functions

In [0]:
"""
Helper functions definition. Used to load and display silver tables, and to write gold tables.
"""
def load_silver_table(table_name):
    # Load table
    df = spark.table(f"project.silver.{table_name}")
    display(df.limit(10))

    # Ensure proper schema
    df.printSchema()
    return df

def write_gold_table(df, table_name):
    (
    df.write
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .format("delta")
          .saveAsTable(f"project.gold.{table_name}")
    )

    display(spark.table(f"project.gold.{table_name}").limit(10))
    print(f"Table '{table_name}' written in gold schema.")

## Dim_date

In [0]:
# Load sales and targets tables (to be modelled as fact)
df_sales = load_silver_table("sales")
df_targets = load_silver_table("targets")

In [0]:
# Find min and max dates across future fact tables
min_sales_order_date = df_sales.select("order_date").agg({"order_date": "min"}).collect()[0][0]
min_targets_date = df_targets.select("target_month").agg({"target_month": "min"}).collect()[0][0]
min_date = min(min_sales_order_date, min_targets_date)

max_sales_order_date = df_sales.select("order_date").agg({"order_date": "max"}).collect()[0][0]
max_targets_date = df_targets.select("target_month").agg({"target_month": "max"}).collect()[0][0]
max_date = max(max_sales_order_date, max_targets_date)

print(f"Minimum sales order date: {min_sales_order_date}")
print(f"Minimum target order date: {min_sales_order_date}")
print(f"Minimum dataset date: {min_date}")

print(f"\nMaximum sales order date: {max_sales_order_date}")
print(f"Maximum target order date: {max_targets_date}")
print(f"Maximum dataset date: {max_date}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# Define a sequence of dates in data range
df_date = spark.sql(f"""
                    SELECT 
                        sequence(to_date('{min_date}'),
                                    to_date('{max_date}'),
                                    interval 1 day) as date_range
                    """)

# Create a date dimension. Each date of the sequence is a row
dim_date = df_date.select(F.explode(col("date_range")).alias("date"))

# Get dim_date fields
dim_date = dim_date.select(
                    F.date_format(col("date"), "yyyyMMdd").cast("int").alias("date_key"),
                    col("date"),
                    F.day(col("date")).alias("day"),
                    F.dayofweek(col("date")).alias("day_of_week"),
                    F.month(col("date")).alias("month"), 
                    F.date_format(col("date"), "MMMM").alias("month_name"),
                    F.quarter(col("date")).alias("quarter"),
                    F.year(col("date")).alias("year"),
                    F.weekofyear(col("date")).alias("week_of_year"),
                    F.when(F.dayofweek(col("date")).isin(1,7), 1).otherwise(0).alias("isWeekend")
)

# Add holiday columns (flag and name)
dim_date = dim_date.withColumn(
        "holiday_name",
        F.when((col("month")==1) & (col("day")==1), "New Years Day")
        .when((col("month")==2) & (col("day")==14), "Valentines Day")
        .when((col("month")==3) & (col("day")==17), "St Patricks Day")
        .when((col("month")==7) & (col("day")==4), "Independence Day")
        .when((col("month")==10) & (col("day")==31), "Halloween")
        .when((col("month")==11) & (col("day")==26), "Thanksgiving")
        .when((col("month")==12) & (col("day")==24), "Christmas Eve")
        .when((col("month")==12) & (col("day")==25), "Christmas Day")
        .when((col("month")==12) & (col("day")==31), "New Years Eve").otherwise(F.lit("N/A"))) \
    .withColumn("isHoliday",
        F.when(col("holiday_name") == "N/A", 0).otherwise(1)
    )

# Sanity check
display(dim_date.limit(15))

In [0]:
# Write dim_date to gold schema
write_gold_table(dim_date, "dim_date")

## Dim_salesperson

In [0]:
# Load salesperson
df_salesperson = load_silver_table("salesperson")

In [0]:
# Define employee key as surrogate key
df_salesperson = df_salesperson.withColumnRenamed("employee_key", "sur_employee_key")

# Write in gold schema
write_gold_table(df_salesperson, "dim_salesperson")

## Fact_targets

In [0]:
# Load targets from silver
load_silver_table("targets")

In [0]:
# Define fact_targets
fact_targets = spark.sql("""
                       SELECT
                           dsp.sur_employee_key,
                           dd.date_key,
                           st.target,
                           st.currency
                       FROM project.silver.targets st
                       JOIN project.gold.dim_salesperson dsp
                        ON st.employee_id = dsp.employee_id
                       JOIN project.gold.dim_date dd
                        ON st.target_month = dd.date
                       """)

# Write fact_targets to gold schema
write_gold_table(fact_targets, "fact_targets")

In [0]:
%sql
DESCRIBE project.gold.fact_sales;

# Control Checks

To guarantee the reliability of the Gold layer, a series of data quality control checks are conducted. These validations ensure foreign key integrity, identify invalid or missing measure values, and verify that the fact table maintains its intended granularity. Collectively, these checks confirm that the curated dataset is consistent, accurate, and prepared for analytical tasks.

# Correct NULL surrogates key check

In [0]:
%sql
SELECT *
FROM project.gold.fact_sales
WHERE sur_order_date_key IS NULL
   OR sur_product_key IS NULL
   OR sur_reseller_key IS NULL
   OR sur_employee_key IS NULL
   OR sur_territory_key IS NULL;

# Negative values control

In [0]:
%sql
SELECT *
FROM project.gold.fact_sales
WHERE quantity < 0
   OR sales_amount < 0
   OR unit_price < 0
   OR total_cost < 0;

# Margin consistency check

In [0]:
%sql
SELECT *
FROM project.gold.fact_sales
WHERE margin_amount != sales_amount - total_cost;

# Duplicate grain check

In [0]:
%sql
SELECT
    sales_order_number,
    sur_product_key,
    sur_reseller_key,
    COUNT(*) AS cnt
FROM project.gold.fact_sales
GROUP BY
    sales_order_number,
    sur_product_key,
    sur_reseller_key
HAVING COUNT(*) > 1;

# Currency consistency check

In [0]:
%sql
SELECT DISTINCT currency
FROM project.gold.fact_sales;

# Check for NULL measures

In [0]:
%sql
SELECT *
FROM project.gold.fact_sales
WHERE quantity IS NULL
   OR sales_amount IS NULL
   OR unit_price IS NULL;

# validation summary

In [0]:
%sql
SELECT 'null_bk' AS check_name, COUNT(*) AS issue_count
FROM project.gold.fact_sales
WHERE sur_product_key IS NULL

UNION ALL

SELECT 'negative_sales', COUNT(*)
FROM project.gold.fact_sales
WHERE sales_amount < 0

UNION ALL

SELECT 'null_sales_amount', COUNT(*)
FROM project.gold.fact_sales
WHERE sales_amount IS NULL

UNION ALL

SELECT 'invalid_margin', COUNT(*)
FROM project.gold.fact_sales
WHERE margin_amount != sales_amount - total_cost;

All validation checks found no inconsistencies, confirming that the Gold layer is structurally sound and reliable for analysis. The dimensional model is now prepared for downstream analytics and BI applications.